In [ ]:
import xlwings as xw
import os
import shutil
import pandas as pd
import numpy as np
import uuid
import time
from glob import glob
import matplotlib
matplotlib.use('Agg')  
import matplotlib.pyplot as plt

names = ['raw20190627SY','raw20190709SY','raw20190728SY','raw20190803SY','raw20190810SY','raw20190818SY','raw20190823SY','raw20190907SY','raw20200615SY']

app = xw.App(visible=False)
app.display_alerts = False
app.screen_updating = False

def safe_array(series, default=0.0):
    values = pd.to_numeric(series[1:], errors='coerce')  
    values = values.fillna(default)                      
    return np.array(values, dtype=np.float32)


def arrhenius_correction(param_T, Tleaf, Ea):
    R = 8.314
    T = Tleaf + 273.15
    T25 = 298.15
    return param_T * np.exp(Ea * (T25 - T) / (R * T * T25))


for name in names:

    template_path = r'ACi_CurveFitting_10.xls'
    base_input_folder = r'../data/soybean2019'
    base_output_folder = r'../data/soybean2019_result'

    input_folder = os.path.join(base_input_folder, name)  
    output_folder = os.path.join(base_output_folder, f'Ellsworth_2004_{name}')
    os.makedirs(output_folder, exist_ok=True)

    files = glob(os.path.join(input_folder, '*.xlsx'))
    files.sort()

    result_list = []
    skipped_files = []

    for input_file in files:
        temp_file = os.path.join(output_folder, f'tmp_{uuid.uuid4().hex[:8]}.xls')
        shutil.copyfile(template_path, temp_file)

        wb = None
        try:
            wb = app.books.open(temp_file)
            sheet = wb.sheets['Calculations']

            xlsx = pd.read_excel(input_file, skiprows=14)
            ci = safe_array(xlsx['Ci'])
            A = safe_array(xlsx['A'])
            PPFD = safe_array(xlsx['Qin'])
            Temp = safe_array(xlsx['Tleaf'])

            sheet.range('B34:F52').clear_contents()

            sheet.range('B34:F52').clear_contents()
            sheet.range('B34').options(transpose=True).value = ci
            sheet.range('C34').options(transpose=True).value = A
            sheet.range('D34').options(transpose=True).value = PPFD
            sheet.range('E34').options(transpose=True).value = Temp
          
            if len(ci) > 0:  
                f_values = [1013.25] * len(ci)
                sheet.range('F34').options(transpose=True).value = f_values

            wb.macro('Ellsworth')()
            
            Vcmax = sheet.range('B28').value
            J = sheet.range('C28').value
            Rd = sheet.range('D28').value
            
            Tleaf_avg = np.mean(Temp)
            print(Tleaf_avg)

            atmpress = 101.325

            GammaS = (4.29 + 0.2268 * (Tleaf_avg - 25) + 0.00463 * (Tleaf_avg - 25)**2) * (atmpress / 101.3)

            Ci_raw = sheet.range('B34:B57').value
            Ci = np.array([float(v) for v in Ci_raw if v is not None and not pd.isna(v) and str(v).strip() != ''])

            A_obs_raw = sheet.range('C34:C57').value
            A_obs = np.array([float(v) for v in A_obs_raw if v is not None and not pd.isna(v) and str(v).strip() != ''])

            Aj_raw = sheet.range('I34:I57').value
            Aj = np.array([float(v) for v in Aj_raw if v is not None and not pd.isna(v) and str(v).strip() != ''])

            Ac_raw = sheet.range('J34:J57').value
            Ac = np.array([float(v) for v in Ac_raw if v is not None and not pd.isna(v) and str(v).strip() != ''])
            
            n = len(A) + 34 - 1
            An_raw = sheet.range(f'AO34:AO{n}').value
            An = np.array([float(v) for v in An_raw if v is not None and not pd.isna(v) and str(v).strip() != ''])
            
            Vcmax25 = arrhenius_correction(Vcmax, Tleaf_avg, Ea=65330)
            J25 = arrhenius_correction(J, Tleaf_avg, Ea=43900)
            R2 = 1 - np.sum((A_obs - An)**2) / np.sum((A_obs - np.mean(A_obs))**2)
            RMSE = np.sqrt(np.mean((A_obs - An) ** 2))

            result = {
                'filename': os.path.basename(input_file),
                'Vcmax': Vcmax,
                'J':     J,
                'Rd':    sheet.range('D28').value,
                'GammaS': round(GammaS, 4), 
                'RMSE': round(RMSE, 4),
                'R2':  round(R2, 4),
                'Vcmax25': round(Vcmax25, 2),
                'J25': round(J25, 2),
            }
            result_list.append(result)
            print(f"✅: {input_file}")

           
            df_plot = pd.DataFrame({
                'Ci': ci,
                'A': A,
                'An': An,
                'Ac': Ac,
                'Aj': Aj
            })

        except Exception as e:
            print(f"❌: {input_file} → {e}")
        finally:
            if wb:
                wb.close()
            time.sleep(0.2)
            if os.path.exists(temp_file):
                os.remove(temp_file)

    df_result = pd.DataFrame(result_list)
    df_result.to_csv(os.path.join(output_folder, 'result.csv'), index=False)
    print(f"📁: {os.path.join(output_folder, 'result.csv')}")

    if skipped_files:
        pd.DataFrame(skipped_files, columns=['skipped_file']).to_csv(
            os.path.join(output_folder, 'skipped_files.csv'), index=False)
        print(f"⚠️: skipped_files.csv")

#close
app.quit()


27.326796
✅: ../data/soybean2019\raw20190627SY\2019-06-27-ACi-top2.xlsx
27.208937
✅: ../data/soybean2019\raw20190627SY\2019-06-27-ACi-top3.xlsx
28.065968
✅: ../data/soybean2019\raw20190627SY\2019-06-27-ACi-top4.xlsx
27.691832
✅: ../data/soybean2019\raw20190627SY\2019-06-27-ACi-top5.xlsx
📁: ../data/soybean2019_result\01Ellsworth_2004_raw20190627SY\result.csv
27.158709
✅: ../data/soybean2019\raw20190709SY\2019-07-09-ACi-top1.xlsx
27.130291
✅: ../data/soybean2019\raw20190709SY\2019-07-09-ACi-top2.xlsx


KeyboardInterrupt: 